In [37]:
import ollama

try:
    models = ollama.list()

    print("Ollama connection successful!")
    print("\nAvailable models:")

    for model in models["models"]:
        print("-", model["model"])

except Exception as e:
    print("Ollama connection failed!")
    print("Error:", e)

Ollama connection successful!

Available models:
- gemma3:1b
- nomic-embed-text:latest
- hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF:latest
- hf.co/CompendiumLabs/bge-base-en-v1.5-gguf:latest
- llama3.2:1b
- gemma4:latest


In [38]:
from langchain_ollama import ChatOllama

# Initialize local Gemma 4 model
llm = ChatOllama(
    model="gemma4:latest",
    temperature=0
)

# Test the model
response = llm.invoke(
    "Explain what a multi-agent AI system is in simple terms."
)

print("Gemma 4 Response:\n")
print(response.content)

Gemma 4 Response:

This is a complex topic, but the easiest way to understand it is to think of it as **a team of smart, independent workers.**

Instead of having one giant, all-knowing AI brain trying to solve a problem, a multi-agent system breaks the problem down and gives smaller, specialized AI "workers" (the agents) the job.

Here is a simple breakdown of the concept, starting with the basics.

---

## 🤖 1. The Basic Building Block: The "Agent"

First, you have to understand what a single **Agent** is.

An agent is simply a piece of software (or a robot) that has three things:

1.  **A Goal:** It knows what it needs to achieve (e.g., "Find the fastest route," or "Collect 10 items").
2.  **Senses:** It can perceive its environment (e.g., "I see a red light," or "The temperature is 70 degrees").
3.  **Actions:** It can take steps to achieve its goal (e.g., "Turn left," or "Increase the heat").

**Think of a single agent as a smart drone.** It has a mission (fly to point X) and it c

In [39]:
from crewai import LLM

# Initialize Gemma 4 for CrewAI
crew_llm = LLM(
    model="ollama/gemma4:latest",
    base_url="http://localhost:11434",
    temperature=0.2
)

# Test the model
response = crew_llm.call(
    "Explain machine learning in two sentences."
)

print(response)

Machine learning is a field of artificial intelligence that allows computers to learn patterns and make decisions directly from data, rather than being explicitly programmed for every task. By training on vast amounts of information, the machine builds predictive models that can identify trends, classify data, and make informed predictions about new, unseen inputs.


In [40]:
# PDF processing
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings and vector database
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# LangChain prompt
from langchain_core.prompts import ChatPromptTemplate

# CrewAI
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool

# Tavily web search
from tavily import TavilyClient

# Utilities
import os
from getpass import getpass

In [41]:
# Initialize embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Test embedding generation
test_embedding = embeddings.embed_query(
    "Artificial Intelligence in healthcare"
)

print("Embedding dimensions:", len(test_embedding))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2144.98it/s]


Embedding dimensions: 384


In [42]:
# Load PDF document
loader = PyPDFLoader("ESRIS.pdf")

# Extract text
documents = loader.load()

print("Total Pages:", len(documents))
print("\nFirst Page Content:\n")

print(documents[0].page_content[:1000])

Total Pages: 14

First Page Content:

Received 13 May 2026, accepted 8 June 2026, date of publication 16 June 2026, date of current version 24 June 2026.
Digital Object Identifier 10.1109/ACCESS.2026.3704255
Energy Selective Reconfigurable Intelligent
Surface: A Power-Adaptive Paradigm for Secure
6G Communications
IC-PYO HONG
 1, (Member, IEEE), BISWARUP RANA
 2, (Senior Member, IEEE),
AND SUNG-SIL CHO1
1Department of Smart Information Technology Engineering, Kongju National University, Cheonan 31080, South Korea
2Smart Natural Space Research Centre, Kongju National University, Cheonan 31080, South Korea
Corresponding author: Ic-Pyo Hong (iphong@kongju.ac.kr)
This work was supported in part by the Priority Research Center Program through the National Research Foundation under Grant
RS-2019-NR040074; in part by Korea Basic Science Institute (National Research Facilities and Equipment Center) grant funded
by the Ministry of Education under Grant RS-2022-NF000835; in part by the Institute

In [43]:

# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
    add_start_index=True
)

# Split PDF into chunks
chunks = text_splitter.split_documents(documents)

print("Total PDF Pages:", len(documents))
print("Total Text Chunks:", len(chunks))

# Display one sample chunk
print("\nSample Chunk:\n")
print(chunks[0].page_content[:500])

print("\nMetadata:")
print(chunks[0].metadata)


Total PDF Pages: 14
Total Text Chunks: 93

Sample Chunk:

Received 13 May 2026, accepted 8 June 2026, date of publication 16 June 2026, date of current version 24 June 2026.
Digital Object Identifier 10.1109/ACCESS.2026.3704255
Energy Selective Reconfigurable Intelligent
Surface: A Power-Adaptive Paradigm for Secure
6G Communications
IC-PYO HONG
 1, (Member, IEEE), BISWARUP RANA
 2, (Senior Member, IEEE),
AND SUNG-SIL CHO1
1Department of Smart Information Technology Engineering, Kongju National University, Cheonan 31080, South Korea
2Smart Natural Spac

Metadata:
{'producer': 'pdfTeX-1.40.24; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-06-23T10:10:02+05:30', 'moddate': '2026-06-24T09:24:54-04:00', 'ieee article id': '11568470', 'trapped': 'False', 'ieee issue id': '11323511', 'subject': 'IEEE Access;2026;14; ;10.1109/ACCESS.2026.3704255', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1

In [44]:

# Create FAISS vector database
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("FAISS database created successfully!")
print("Total chunks stored:", vectorstore.index.ntotal)


FAISS database created successfully!
Total chunks stored: 93


In [45]:
# Search query
query = "What is the main topic of this document?"

# Retrieve relevant chunks
results = vectorstore.similarity_search(
    query,
    k=3
)

# Display search results
for i, doc in enumerate(results):

    print(f"\nResult {i+1}:")
    print("Content:", doc.page_content[:500])
    print("Source:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page", 0) + 1)


Result 1:
Content: 2026 The Authors. This work is licensed under a Creative Commons Attribution 4.0 License.
For more information, see https://creativecommons.org/licenses/by/4.0/ VOLUME 14, 2026
Source: ESRIS.pdf
Page: 1

Result 2:
Content: he is currently a Professor. His research interests include numerical
techniques in electromagnetics, periodic electromagnetic structures, and
their applications in wireless communications.
BISWARUP RANA (Senior Member, IEEE)
received the Ph.D. degree from Indian Insti-
tute of Engineering Science and Technology,
Shibpur, India, in 2017. He was a Postdoctoral
Researcher with Seoul National University of Sci-
ence and Technology, South Korea. He is currently
with Kongju National University, South
Source: ESRIS.pdf
Page: 14

Result 3:
Content: grated waveguide antennas, phased array antennas,
dielectric resonator antennas, implantable antennas, transmitarray, reconfig-
urable intelligent surfaces, and the design of metasurfaces based on AI/ML.
He is

In [46]:

# Initialize MMR retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 12,
        "lambda_mult": 0.7
    }
)

# Test research question
question = "What are the structural limitations of existing RIS technologies?"

# Retrieve relevant chunks
retrieved_docs = retriever.invoke(question)

print("Retrieved Documents:", len(retrieved_docs))

# Display retrieved information
for i, doc in enumerate(retrieved_docs):

    print(f"\nResult {i+1}")

    print("Page:", doc.metadata.get("page", 0) + 1)

    print("Content:")
    print(doc.page_content[:700])

    print("-" * 60)


Retrieved Documents: 4

Result 1
Page: 3
Content:
such as communication quality improvement, beamforming,
or channel environment reconfiguration [30]. Fig. 1 shows the
limitations of the existing RIS. It also shows the capabilities
of our proposed ES–RIS.
III. NOVELTY JUSTIFICATION AND COMPARISON WITH
EXISTING RIS
Conventional passive types of RIS have fixed beamforming
capabilities; however, this type of RIS does not have capabil-
ities to protect from HPM/EMP signals or other functions.
Active RIS has dynamic beam-forming capabilities. How-
ever, future networks require additional functionality from the
RIS. There are other types of RIS, like zeRIS/EH-RIS, which
have beam steering and self-powering capabilities. Unlike
previously proposed
------------------------------------------------------------

Result 2
Page: 3
Content:
and are evaluated as a low-power, low-cost solution for
realizing a smart radio environment [25]. However, the
reconfigurability of current RIS prototypes is gen

In [47]:
# Display all retrieved chunks completely
for i, doc in enumerate(retrieved_docs):

    print(f"\n{'='*70}")
    print(f"RESULT {i+1}")
    print(f"PAGE: {doc.metadata.get('page', 0) + 1}")
    print(f"{'='*70}")

    print(doc.page_content)


RESULT 1
PAGE: 3
such as communication quality improvement, beamforming,
or channel environment reconfiguration [30]. Fig. 1 shows the
limitations of the existing RIS. It also shows the capabilities
of our proposed ES–RIS.
III. NOVELTY JUSTIFICATION AND COMPARISON WITH
EXISTING RIS
Conventional passive types of RIS have fixed beamforming
capabilities; however, this type of RIS does not have capabil-
ities to protect from HPM/EMP signals or other functions.
Active RIS has dynamic beam-forming capabilities. How-
ever, future networks require additional functionality from the
RIS. There are other types of RIS, like zeRIS/EH-RIS, which
have beam steering and self-powering capabilities. Unlike
previously proposed RISs, where it was assumed that the
impinging power on the RIS would be steady, the proposed

RESULT 2
PAGE: 3
and are evaluated as a low-power, low-cost solution for
realizing a smart radio environment [25]. However, the
reconfigurability of current RIS prototypes is generally
ba

In [48]:
# Check all retrieved chunks without excessive output
for i, doc in enumerate(retrieved_docs):

    print(f"\nRESULT {i+1}")
    print("Page:", doc.metadata.get("page", 0) + 1)
    print("Chunk length:", len(doc.page_content))

    print("\nBeginning:")
    print(doc.page_content[:350])

    print("\nEnding:")
    print(doc.page_content[-250:])

    print("-" * 60)


RESULT 1
Page: 3
Chunk length: 793

Beginning:
such as communication quality improvement, beamforming,
or channel environment reconfiguration [30]. Fig. 1 shows the
limitations of the existing RIS. It also shows the capabilities
of our proposed ES–RIS.
III. NOVELTY JUSTIFICATION AND COMPARISON WITH
EXISTING RIS
Conventional passive types of RIS have fixed beamforming
capabilities; however, this

Ending:
ctionality from the
RIS. There are other types of RIS, like zeRIS/EH-RIS, which
have beam steering and self-powering capabilities. Unlike
previously proposed RISs, where it was assumed that the
impinging power on the RIS would be steady, the proposed
------------------------------------------------------------

RESULT 2
Page: 3
Chunk length: 747

Beginning:
and are evaluated as a low-power, low-cost solution for
realizing a smart radio environment [25]. However, the
reconfigurability of current RIS prototypes is generally
based on FPGA controllers, which require significant power
consu

In [49]:
# Prepare context from retrieved PDF chunks
context = ""

for i, doc in enumerate(retrieved_docs):

    page = doc.metadata.get("page", 0) + 1

    context += f"\nSource: ESRIS.pdf | Page: {page}\n"
    context += doc.page_content
    context += "\n"

# Create prompt for grounded answer generation
prompt = f"""
You are an academic research assistant.

Answer the question using ONLY the provided PDF context.

Instructions:
1. Use clear and simple academic language.
2. Do not add information from outside the context.
3. Organize the answer using bullet points.
4. Mention the PDF page number for each relevant point.
5. If the context does not contain enough information, say so.

PDF CONTEXT:
{context}

QUESTION:
What are the structural limitations of existing RIS technologies?

ANSWER:
"""

# Generate answer using local Gemma 4
rag_llm = ChatOllama(
    model="gemma4:latest",
    temperature=0
)

response = rag_llm.invoke(prompt)

print("========== RAG GENERATED ANSWER ==========\n")
print(response.content)

========== RAG GENERATED ANSWER ==========

The structural limitations of existing RIS technologies include the following:

*   **Functional Constraints of Passive RIS:** Conventional passive types of RIS possess fixed beamforming capabilities and lack the ability to protect against HPM/EMP signals or other necessary functions (Page 3).
*   **Need for Enhanced Functionality:** While active RIS offers dynamic beam-forming capabilities, future network requirements necessitate additional functionality from the RIS (Page 3).
*   **Power and Deployment Issues of Prototypes:**
    *   The reconfigurability of current RIS prototypes is typically based on FPGA controllers, which results in significant power consumption and requires the RIS to be continuously connected to the power grid (Page 3).
    *   For large-scale RIS deployment, the planning and maintenance costs associated with cable installation increase exponentially (Page 3).
    *   Using single-use batteries for power supply is not

In [50]:
# Test another question using the same RAG pipeline

question = "What are the main features and advantages of the proposed ES-RIS?"

# Retrieve relevant PDF chunks
retrieved_docs = retriever.invoke(question)

# Prepare context
context = ""

for doc in retrieved_docs:

    page = doc.metadata.get("page", 0) + 1

    context += f"\nSource: ESRIS.pdf | Page: {page}\n"
    context += doc.page_content + "\n"

# Generate answer
prompt = f"""
Answer the question using ONLY the provided PDF context.

Use clear academic language.
Organize the answer into bullet points.
Include page references.
Do not invent information.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

response = rag_llm.invoke(prompt)

print("========== RAG TEST 2 ==========\n")
print(response.content)

========== RAG TEST 2 ==========

The main features and advantages of the proposed ES-RIS include:

*   **Enhanced Security:** The system possesses the capability to suppress harmful signals, providing security benefits (Page 11).
*   **Cognitive and Adaptive Functionality:** As a cognitive electromagnetic surface, the ES-RIS utilizes AI and machine learning to adapt to its environment. It achieves this by taking into account signal strength, interference, and channel conditions (Page 11).
*   **Multi-Mode Operation:** The ES-RIS is capable of operating in various modes, specifically communication mode, security mode, and energy harvesting mode. A sensing mode may also be incorporated in the future (Page 11).
*   **Autonomous Control:** A significant feature is the automatic mode switching, which allows the system to operate without requiring external control (Page 11).


In [51]:
# Create prompt template
prompt = ChatPromptTemplate.from_template("""
You are an intelligent research assistant.

Answer the question using only the provided document context.

If the answer is not available in the context,
clearly say that the information is not available.

Include the source PDF name and page number in your answer.

Context:
{context}

Question:
{question}

Answer:
""")

In [52]:
# Generate final prompt
final_prompt = prompt.format(
    context=context,
    question=question
)

# Generate answer using local Gemma 4
response = llm.invoke(final_prompt)

print("\nResearch Assistant Answer:\n")
print(response.content)


Research Assistant Answer:

The main features and advantages of the proposed ES-RIS include:

*   **Security:** It can suppress harmful signals (ESRIS.pdf | Page: 11).
*   **Cognitive Electromagnetic Surface:** It is a new concept that works with AI and machine learning, allowing it to adapt to the environment by taking into account signal strength, interference, and channel conditions (ESRIS.pdf | Page: 11).
*   **Operational Modes:** ES-RIS can operate in different modes, such as communication mode, security mode, and energy harvesting mode (ESRIS.pdf | Page: 11).
*   **Automatic Switching:** It features automatic mode switching, which operates without any external control (ESRIS.pdf | Page: 11).


In [53]:
# Research question
question = "What are the limitations of existing RIS technologies in 6G communications?"

# Retrieve relevant information
results = vectorstore.similarity_search(
    question,
    k=5
)

for i, doc in enumerate(results):

    print(f"\nResult {i+1}")

    print("Page:", doc.metadata.get("page", 0) + 1)

    print(doc.page_content[:500])


Result 1
Page: 2
I.-P. Hong et al.: ES-RIS: A Power-Adaptive Paradigm for Secure 6G Communications
FIGURE 1. Overview of the need for ES-RIS — Limitations of existing technologies (left) and ES-RIS solution capabilities (right).
and greater substitutability. Importantly, 6G will extend
beyond the scope of traditional communication networks [2].
Because of these advancements, new types of applications
are expected to be introduced in 6G communications, such
as the metaverse, Industry 5.0, Smart Grid 2.0, Collabor

Result 2
Page: 3
A unified solution can greatly simplify these challenges and
enhance overall performance. Hence, developing flexible and
reconfigurable surfaces is critically important for establishing
a powerful and expandable 6G infrastructure. The main con-
tributions of this paper are (1) proposal of the ES-RIS concept
combining RIS + ESS, (2) analytical impedance model,
(3) dual-mode validation (simulation), and (4) research
roadmap for 6G.
II. STRUCTURAL LIMITATIONS OF

In [54]:

@tool("PDF Knowledge Search")
def pdf_search(query: str) -> str:
    """
    Search the uploaded PDF using FAISS MMR retrieval.
    Return relevant content with source and page references.
    """

    # Retrieve relevant chunks using the existing MMR retriever
    results = retriever.invoke(query)

    context = ""

    for i, doc in enumerate(results):

        # Get PDF page number
        page = doc.metadata.get("page", 0) + 1

        # Get source PDF name
        source = os.path.basename(
            doc.metadata.get("source", "Unknown")
        )

        context += f"""
Result: {i+1}
Source: {source}
Page: {page}

Content:
{doc.page_content}

--------------------------------
"""

    if not context:
        return "No relevant information found in the PDF."

    return context


In [55]:
# Enter your Tavily API key securely
api_key = getpass("Enter your Tavily API key: ").strip()

if not api_key:
    raise ValueError("Tavily API key is missing.")

os.environ["TAVILY_API_KEY"] = api_key

print("Tavily API key configured successfully!")

Tavily API key configured successfully!


In [56]:
# Initialize Tavily client
tavily = TavilyClient(
    api_key=os.environ["TAVILY_API_KEY"]
)

print("Tavily connected successfully!")

Tavily connected successfully!


In [57]:
# Test external web search
response = tavily.search(
    query="Latest developments in 6G communication technology",
    max_results=3
)

for i, result in enumerate(response["results"]):

    print(f"\nResult {i+1}")

    print("Title:", result["title"])

    print("URL:", result["url"])

    print("Content:", result["content"][:500])


Result 1
Title: A Review of Recent Developments in 6G Communications Systems
URL: https://www.mdpi.com/2673-4591/59/1/167
Content: - The advancement of AI chips has resulted in a breakthrough in which hardware can be isolated from transceiver algorithms, and they can configure and update themselves based on environmental inputs.
  - Intelligent radio today is expected to solve many problems of accurate channel modeling, dynamic spectrum access, improved network deployment, optimization, and autonomous orchestration that lead to malicious attacks/activities during communication systems. [...] The 6G 2030 network concepts inc

Result 2
Title: The future of 6G: transformative potential & how it will change everyday life
URL: https://www.thalesgroup.com/en/news-centre/insights/enterprise/mobile-communications/6g-under-construction-what-are-latest
Content: In December 2023 the ITU published its framework for 6G (which is also called IMT-2030) technical standards. In March 2024, the 3GPP pu

In [58]:
@tool("Web Research Search")
def web_search(query: str) -> str:
    """Search the web using Tavily and return research information with source URLs."""

    # Perform external web search
    response = tavily.search(
        query=query,
        search_depth="advanced",
        max_results=5
    )

    context = ""

    for i, result in enumerate(response["results"]):

        context += f"""
Result: {i+1}
Title: {result['title']}
URL: {result['url']}
Content: {result['content']}

"""

    return context if context else "No relevant web information found."

In [59]:
# Test the web research tool
web_results = web_search.run(
    "Recent developments in 6G communication technology"
)

print(web_results)


Result: 1
Title: 6G: Key Hardware Technologies and Future Development Roadmap | IDTechEx Research Article
URL: https://www.idtechex.com/en/research-article/6g-key-hardware-technologies-and-future-development-roadmap/32034
Content: The development of 6G technology has rapidly progressed since 2017. China and the US initiated 6G research early on, highlighting global interest in next-generation wireless technology. Collaborative efforts like the US Next G Alliance, Japan's B5G consortium, and the EU's Hexa-X project have fostered international innovation.

Significant advancements include China launching its first 6G satellite and companies like Apple integrating satellite-based services in their devices. According to IDTechEx, 6G technology will gradually emerge around 2028, with commercialization expected by 2030. The 3GPP standardization process, beginning in 2026, will be crucial for its adoption. [...] Although global spectrum allocations have yet to exceed 275 GHz, the range from 

In [60]:
document_agent = Agent(

    role="Document Research Specialist",

    goal="""
    Retrieve accurate information from PDF documents
    and answer research questions with source references.
    """,

    backstory="""
    You are an experienced document research specialist.

    You analyze technical PDF documents, identify relevant
    information and provide accurate answers with page references.

    You never invent document information.
    """,

    llm=crew_llm,

    tools=[pdf_search],

    verbose=True,

    allow_delegation=False
)

print("Document Research Agent created successfully!")

Document Research Agent created successfully!


In [61]:
web_agent = Agent(

    role="Web Research Specialist",

    goal="""
    Search the internet for reliable and recent information
    related to the research question.
    Collect technical findings and source URLs.
    """,

    backstory="""
    You are an expert web research specialist.

    You use Tavily to search for technical research papers,
    articles and industry reports.

    You identify relevant findings and preserve source URLs.

    Never invent facts or references.
    """,

    llm=crew_llm,

    tools=[web_search],

    verbose=True,

    allow_delegation=False
)

print("Web Research Agent created successfully!")

Web Research Agent created successfully!


In [62]:
analyst_agent = Agent(

    role="Senior Research Analyst",

    goal="""
    Analyze and compare information collected from PDF
    documents and web research.
    Identify technical insights and research gaps.
    """,

    backstory="""
    You are an experienced technical research analyst.

    You analyze research papers, technical PDFs and web articles.

    You compare findings, identify similarities and differences,
    and produce meaningful technical insights.

    You maintain source references and avoid unsupported claims.
    """,

    llm=crew_llm,

    verbose=True,

    allow_delegation=False
)

print("Research Analyst Agent created successfully!")

Research Analyst Agent created successfully!


In [63]:
report_agent = Agent(

    role="Professional Research Report Writer",

    goal="""
    Create a detailed and well-structured research report
    using PDF findings and web research.
    """,

    backstory="""
    You are a professional technical report writer.

    You prepare accurate and readable research reports.

    You preserve PDF page references and web source URLs.

    You never invent citations or unsupported information.
    """,

    llm=crew_llm,

    verbose=True,

    allow_delegation=False
)

print("Report Writer Agent created successfully!")

Report Writer Agent created successfully!


In [64]:
question = """
What are the structural limitations of existing RIS technologies
and what recent developments may address them?
"""

print("Research Question:")
print(question)

Research Question:

What are the structural limitations of existing RIS technologies
and what recent developments may address them?



In [65]:
research_task = Task(

    description="""
    Answer the following research question:

    {question}

    Use the PDF Knowledge Search tool to retrieve
    relevant information from the uploaded document.

    Explain the important findings clearly.

    Include source document names and page references.

    Do not invent information that is not present in the document.
    """,

    expected_output="""
    A structured research answer containing:

    1. Introduction to the research topic.
    2. Important findings.
    3. Technical explanations.
    4. PDF source name.
    5. Correct page references.
    """,

    agent=document_agent
)

print("Document Research Task created successfully!")

Document Research Task created successfully!


In [66]:
web_research_task = Task(

    description="""
    Conduct web research on the following topic:

    {question}

    Instructions:

    1. Use the Web Research Search tool.
    2. Search for recent and relevant information.
    3. Identify important technical developments.
    4. Collect findings from reliable sources.
    5. Include source titles and URLs.
    6. Avoid duplicate information.
    7. Do not invent facts or references.
    """,

    expected_output="""
    A structured web research summary containing:

    1. Research topic.
    2. Important findings.
    3. Technical developments.
    4. Supporting explanations.
    5. Source titles.
    6. Source URLs.
    """,

    agent=web_agent
)

print("Web Research Task created successfully!")

Web Research Task created successfully!


In [67]:
analysis_task = Task(

    description="""
    Analyze the findings collected by the Document Research
    Agent and Web Research Agent.

    Research question:
    {question}

    Your responsibilities:

    1. Analyze PDF research findings.
    2. Analyze web research findings.
    3. Compare information from both sources.
    4. Identify important technical developments.
    5. Identify limitations and research gaps.
    6. Highlight similarities and differences.
    7. Preserve PDF page references and web URLs.
    8. Avoid unsupported claims.

    Clearly distinguish PDF-based findings
    from web-based findings.
    """,

    expected_output="""
    A comprehensive research analysis containing:

    1. Summary of PDF findings.
    2. Summary of web findings.
    3. Comparative analysis.
    4. Important technical insights.
    5. Identified research gaps.
    6. PDF source names and page references.
    7. Web source titles and URLs.
    """,

    agent=analyst_agent,

    context=[
        research_task,
        web_research_task
    ]
)

print("Research Analysis Task created successfully!")

Research Analysis Task created successfully!


In [68]:

report_task = Task(

    description="""
    Prepare a concise research report using the outputs
    of the Document Research, Web Research and Analyst agents.

    Research question:
    {question}

    Include these sections:

    1. Title
    2. Executive Summary
    3. Introduction
    4. Key Findings from PDF
    5. Recent Developments from Web Research
    6. Comparative Analysis
    7. Conclusion
    8. References

    Instructions:

    - Keep the report around 400-500 words.
    - Use clear and simple academic language.
    - Preserve PDF page numbers and web source URLs.
    - Use only information from previous tasks.
    - Do not invent facts or references.
    - Avoid unnecessary explanations.
    """,

    expected_output="""
    A complete, concise research report with all
    eight sections and accurate source references.
    """,

    agent=report_agent,

    context=[
        research_task,
        web_research_task,
        analysis_task
    ]
)

print("Research Report Task created successfully!")


Research Report Task created successfully!


In [69]:

# Connect all four agents and their tasks

research_crew = Crew(

    agents=[
        document_agent,
        web_agent,
        analyst_agent,
        report_agent
    ],

    tasks=[
        research_task,
        web_research_task,
        analysis_task,
        report_task
    ],

    process=Process.sequential,

    verbose=True
)

print("IntelliResearch AI Multi-Agent Crew created successfully!")


IntelliResearch AI Multi-Agent Crew created successfully!


In [70]:
# Execute all four agents
result = await research_crew.kickoff_async(
    inputs={
        "question": question
    }
)

print("\n========== INTELLIRESEARCH AI REPORT ==========\n")

print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b8d40a0f-1219-419e-8865-e344bdf13d33                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Answer the following research question:                                                                    │
│                                                                                                                 │
│                                                                                                                 │
│  What are the structural limitations of existing RIS technologies                                               │
│  and what recent developments may address them?                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│      Use the PDF Knowledge Search tool to retrieve                                                              │
│      relevant information from the uploaded document.                                                           │
│                                                                                                                 │
│      Explain the important findings clearly.                                                                    │
│                                                                                                                 │
│      Include source document names and page references.                                                         │
│                                                                                                                 │
│      Do not invent information that is not present in the document.                                             │
│                                                                                                                 │
│  ID: 2460d391-fc6e-495f-bfab-193a7e66bccf                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Document Research Specialist                                                                            │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Answer the following research question:                                                                    │
│                                                                                                                 │
│                                                                                                                 │
│  What are the structural limitations of existing RIS technologies                                               │
│  and what recent developments may address them?                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│      Use the PDF Knowledge Search tool to retrieve                                                              │
│      relevant information from the uploaded document.                                                           │
│                                                                                                                 │
│      Explain the important findings clearly.                                                                    │
│                                                                                                                 │
│      Include source document names and page references.                                                         │
│                                                                                                                 │
│      Do not invent information that is not present in the document.                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: pdf_knowledge_search                                                                                     │
│  Args: {'query': 'structural limitations of existing RIS technologies and recent developments to address        │
│  them'}                                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: pdf_knowledge_search                                                                                     │
│  Output:                                                                                                        │
│  Result: 1                                                                                                      │
│  Source: ESRIS.pdf                                                                                              │
│  Page: 3                                                                                                        │
│                                                                                                                 │
│  Content:                                                                                                       │
│  such as communication quality improvement, beamforming,                                                        │
│  or channel environment reconfiguration [30]. Fig. 1 shows the                                                  │
│  limitations of the existing RIS. It also shows the capabilities                                                │
│  of our proposed ES–RIS.                                                                                        │
│  III. NOVELTY JUSTIFICATION AND COMPARISON WITH                                                                 │
│  EXISTING RIS                                                                                                   │
│  Conventional passive types of RIS have fixed beamforming                                                       │
│  capabilities; however, this type of RIS does not have capabil-                                                 │
│  ities to protect from HPM/EMP signals or other functions.                                                      │
│  Active RIS has dynamic beam-forming capabilities. How-                                                         │
│  ever, future networks require additional functionality from the                                                │
│  RIS. There are other types of RIS, like zeRIS/EH-RIS, which                                                    │
│  have beam steering and self-powering capabilities. Unlike                                                      │
│  previously proposed RISs, where it was assumed that the                                                        │
│  impinging power on the RIS would be steady, the proposed                                                       │
│                                                                                                                 │
│  --------------------------------                                                                               │
│                                                                                                                 │
│  Result: 2                                                                                                      │
│  Source: ESRIS.pdf                                                                                              │
│  Page: 3                                                                                                        │
│                                                                                                                 │
│  Content:                                                                                                       │
│  and are evaluated as a low-power, low-cost solution for                                                        │
│  realizing a smart radio environment [25]. However, the

[Finalize] todos_count=0, todos_with_results=0


... (18 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Prepare a concise research report using the outputs                                                        │
│      of the Document Research, Web Research and Analyst agents.                                                 │
│                                                                                                                 │
│      Research question:                                                                                         │
│                                                                                                                 │
│  What are the structural limitations of existing RIS technologies                                               │
│  and what recent developments may address them?                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│      Include these sections:                                                                                    │
│                                                                                                                 │
│      1. Title                                                                                                   │
│      2. Executive Summary                                                                                       │
│      3. Introduction                                                                                            │
│      4. Key Findings from PDF                                                                                   │
│      5. Recent Developments from Web Research                                                                   │
│      6. Comparative Analysis                                                                                    │
│      7. Conclusion                                                                                              │
│      8. References                                                                                              │
│                                                                                                                 │
│      Instructions:                                                                                              │
│                                                                                                                 │
│      - Keep the report around 400-500 words.                                                                    │
│      - Use clear and simple academic language.                                                                  │
│      - Preserve PDF page numbers and web source URLs.                                                           │
│      - Use only information from previous tasks.                                                                │
│      - Do not invent facts or references.                                                                       │
│      - Avoid unnecessary explanations.                                                                          │
│                                                                                                                 │
│  Agent: Professional Research Report Writer                                                                     │
│                                                        

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b8d40a0f-1219-419e-8865-e344bdf13d33                                                                       │
│  Final Output: # Structural Limitations and Advanced Solutions in Reconfigurable Intelligent Surface (RIS)      │
│  Technologies                                                                                                   │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│                                                                                                                 │
│  Existing Reconfigurable Intelligent Surface (RIS) technologies face significant structural and functional      │
│  limitations, spanning from macro-level deployment challenges to micro-level hardware impairments. Key          │
│  limitations include fixed capabilities, high power consumption, and vulnerability to hardware imperfections    │
│  like phase noise and pixel failure. Recent developments are addressing these constraints through the           │
│  introduction of Active RIS architectures, the integration of Energy Storage Systems (ES-RIS), and the          │
│  application of advanced AI/ML and accurate Electromagnetic                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


========== INTELLIRESEARCH AI REPORT ==========

# Structural Limitations and Advanced Solutions in Reconfigurable Intelligent Surface (RIS) Technologies

## Executive Summary

Existing Reconfigurable Intelligent Surface (RIS) technologies face significant structural and functional limitations, spanning from macro-level deployment challenges to micro-level hardware impairments. Key limitations include fixed capabilities, high power consumption, and vulnerability to hardware imperfections like phase noise and pixel failure. Recent developments are addressing these constraints through the introduction of Active RIS architectures, the integration of Energy Storage Systems (ES-RIS), and the application of advanced AI/ML and accurate Electromagnetic


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [72]:

# Enter the research question
question = input("Enter your research question: ")

# Execute the multi-agent research workflow asynchronously
result = await research_crew.kickoff_async(
    inputs={"question": question}
)

# Save the final research report
with open("IntelliResearch_Report.txt", "w", encoding="utf-8") as file:
    file.write(str(result))

print("\nResearch completed successfully!")
print("\nFinal Research Report:")
print(result)

print("\nReport saved as IntelliResearch_Report.txt")


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b8d40a0f-1219-419e-8865-e344bdf13d33                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Answer the following research question:                                                                    │
│                                                                                                                 │
│      advantages of ris                                                                                          │
│                                                                                                                 │
│      Use the PDF Knowledge Search tool to retrieve                                                              │
│      relevant information from the uploaded document.                                                           │
│                                                                                                                 │
│      Explain the important findings clearly.                                                                    │
│                                                                                                                 │
│      Include source document names and page references.                                                         │
│                                                                                                                 │
│      Do not invent information that is not present in the document.                                             │
│                                                                                                                 │
│  ID: 2460d391-fc6e-495f-bfab-193a7e66bccf                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Document Research Specialist                                                                            │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Answer the following research question:                                                                    │
│                                                                                                                 │
│      advantages of ris                                                                                          │
│                                                                                                                 │
│      Use the PDF Knowledge Search tool to retrieve                                                              │
│      relevant information from the uploaded document.                                                           │
│                                                                                                                 │
│      Explain the important findings clearly.                                                                    │
│                                                                                                                 │
│      Include source document names and page references.                                                         │
│                                                                                                                 │
│      Do not invent information that is not present in the document.                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: pdf_knowledge_search                                                                                     │
│  Args: {'query': 'advantages of ris'}                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#2) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: pdf_knowledge_search                                                                                     │
│  Output:                                                                                                        │
│  Result: 1                                                                                                      │
│  Source: ESRIS.pdf                                                                                              │
│  Page: 3                                                                                                        │
│                                                                                                                 │
│  Content:                                                                                                       │
│  such as communication quality improvement, beamforming,                                                        │
│  or channel environment reconfiguration [30]. Fig. 1 shows the                                                  │
│  limitations of the existing RIS. It also shows the capabilities                                                │
│  of our proposed ES–RIS.                                                                                        │
│  III. NOVELTY JUSTIFICATION AND COMPARISON WITH                                                                 │
│  EXISTING RIS                                                                                                   │
│  Conventional passive types of RIS have fixed beamforming                                                       │
│  capabilities; however, this type of RIS does not have capabil-                                                 │
│  ities to protect from HPM/EMP signals or other functions.                                                      │
│  Active RIS has dynamic beam-forming capabilities. How-                                                         │
│  ever, future networks require additional functionality from the                                                │
│  RIS. There are other types of RIS, like zeRIS/EH-RIS, which                                                    │
│  have beam steering and self-powering capabilities. Unlike                                                      │
│  previously proposed RISs, where it was assumed that the                                                        │
│  impinging power on the RIS would be steady, the proposed                                                       │
│                                                                                                                 │
│  --------------------------------                                                                               │
│                                                                                                                 │
│  Result: 2                                                                                                      │
│  Source: ESRIS.pdf                                                                                              │
│  Page: 3                                                                                                        │
│                                                                                                                 │
│  Content:                                                                                                       │
│  and are evaluated as a low-power, low-cost solution for                                                        │
│  realizing a smart radio environment [25]. However, the

[Finalize] todos_count=0, todos_with_results=0


... (18 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Prepare a concise research report using the outputs                                                        │
│      of the Document Research, Web Research and Analyst agents.                                                 │
│                                                                                                                 │
│      Research question:                                                                                         │
│      advantages of ris                                                                                          │
│                                                                                                                 │
│      Include these sections:                                                                                    │
│                                                                                                                 │
│      1. Title                                                                                                   │
│      2. Executive Summary                                                                                       │
│      3. Introduction                                                                                            │
│      4. Key Findings from PDF                                                                                   │
│      5. Recent Developments from Web Research                                                                   │
│      6. Comparative Analysis                                                                                    │
│      7. Conclusion                                                                                              │
│      8. References                                                                                              │
│                                                                                                                 │
│      Instructions:                                                                                              │
│                                                                                                                 │
│      - Keep the report around 400-500 words.                                                                    │
│      - Use clear and simple academic language.                                                                  │
│      - Preserve PDF page numbers and web source URLs.                                                           │
│      - Use only information from previous tasks.                                                                │
│      - Do not invent facts or references.                                                                       │
│      - Avoid unnecessary explanations.                                                                          │
│                                                                                                                 │
│  Agent: Professional Research Report Writer                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b8d40a0f-1219-419e-8865-e344bdf13d33                                                                       │
│  Final Output: # Research Report: Advantages and Evolution of Reconfigurable Intelligent Surfaces (RIS)         │
│                                                                                                                 │
│  ## 1. Title                                                                                                    │
│  Advantages and Technical Evolution of Reconfigurable Intelligent Surfaces (RIS) in Modern Wireless             │
│  Environments                                                                                                   │
│                                                                                                                 │
│  ## 2. Executive Summary                                                                                        │
│  Reconfigurable Intelligent Surfaces (RIS) represent a significant advancement in wireless communication,       │
│  moving beyond the limitations of conventional passive systems. The primary advantages of RIS include enabling  │
│  dynamic beamforming, enhancing energy efficiency through specialized designs like ES-RIS, and improving        │
│  network resilience via Protective RIS. While these technologies offer substantial performance gains, their     │
│  large-scale deployment is currently constrained by high power consumption from control units and               │
│  exponentially increasing installation costs.                                                                   │
│                                                                                                                 │
│  ## 3. Introduction                                                                                             │
│  This report examines the advantages and technical evolution of Reconfigurable Intelligent Surfaces (RIS).      │
│  Conventional RIS technologies are limited by fixed beamforming capabilities and lack resilience against        │
│  modern wireless challenges. The research investigates how advanced RIS types are addressing these              │
│  limitations, focusing on their operational capabilities, specialized functions, and the commercialization      │
│  hurdles they face.                                                                                             │
│                                                                                                                 │
│  ## 4. Key Findings from PDF                                                                                    │
│  The document highlights the transition from passive to intelligent RIS. Conventional passive RIS lacks         │
│  dynamic capabilities and protection against signals like HPM/EMP [ESRIS.pdf, Page 3]. Advanced types include   │
│  Active RIS, which offers dynamic beam-forming, and zeRIS/EH-RIS, which provide beam steering and               │
│  self-powering [ESRIS.pdf, Page 3]. A key innovation is ES-RIS, which operates without assuming steady          │
│  impinging power and features multifunctional power-adaptive behavior [ESRIS.pdf, Page 3]. Furthermore,         │
│  Energy-aware RIS allows the metasurface to adapt based on signal strength, utilizing nonlinear materials for   │
│  energy-dependent switching [ESRIS.pdf, Page 11]. However, prototypes face significant obstacles, including     │
│  high power consumption and grid dependency due to FPG


Research completed successfully!

Final Research Report:
# Research Report: Advantages and Evolution of Reconfigurable Intelligent Surfaces (RIS)

## 1. Title
Advantages and Technical Evolution of Reconfigurable Intelligent Surfaces (RIS) in Modern Wireless Environments

## 2. Executive Summary
Reconfigurable Intelligent Surfaces (RIS) represent a significant advancement in wireless communication, moving beyond the limitations of conventional passive systems. The primary advantages of RIS include enabling dynamic beamforming, enhancing energy efficiency through specialized designs like ES-RIS, and improving network resilience via Protective RIS. While these technologies offer substantial performance gains, their large-scale deployment is currently constrained by high power consumption from control units and exponentially increasing installation costs.

## 3. Introduction
This report examines the advantages and technical evolution of Reconfigurable Intelligent Surfaces (RIS). Conventio

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
# Save final research report
report_text = str(result)

with open("IntelliResearch_Report.txt", "w", encoding="utf-8") as file:

    file.write(report_text)

print("Research report saved successfully!")

Research report saved successfully!


In [ ]:
print(report_text)

# Structural Limitations and Adaptive Solutions in Reconfigurable Intelligent Surface (RIS) Technologies

## 1. Title
Structural Limitations and Adaptive Solutions in Reconfigurable Intelligent Surface (RIS) Technologies for Next-Generation Wireless Networks

## 2. Executive Summary
Reconfigurable Intelligent Surfaces (RIS) are critical components for realizing efficient and scalable 6G infrastructure. However, existing RIS technologies face significant structural, functional, and operational limitations, including fixed beamforming capabilities, vulnerability to high-power signals, and inherent geometric constraints. This report analyzes these limitations by synthesizing findings from technical literature and broader research analyses. Recent developments, particularly the introduction of the Energy Storage RIS (ES-RIS) concept, are addressing these constraints. ES-RIS integrates energy storage to enable a power-adaptive paradigm, providing both communication enhancement and hardware-